In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report

In [2]:
fan_id00_abnormal = pd.read_csv('/content/drive/MyDrive/Listen_work/fan_csv/fan_id00_abnormal')
fan_id00_normal = pd.read_csv('/content/drive/MyDrive/Listen_work/fan_csv/fan_id00_normal')

In [3]:
fan_id02_abnormal = pd.read_csv('/content/drive/MyDrive/Listen_work/fan_csv/fan_id02_abnormal')
fan_id02_normal = pd.read_csv('/content/drive/MyDrive/Listen_work/fan_csv/fan_id02_normal')

In [4]:
fan_id04_abnormal = pd.read_csv('/content/drive/MyDrive/Listen_work/fan_csv/fan_id04_abnormal')
fan_id04_normal = pd.read_csv('/content/drive/MyDrive/Listen_work/fan_csv/fan_id04_normal')


In [5]:
fan_id06_abnormal = pd.read_csv('/content/drive/MyDrive/Listen_work/fan_csv/fan_id06_abnormal')
fan_id06_normal = pd.read_csv('/content/drive/MyDrive/Listen_work/fan_csv/fan_id06_normal')

In [6]:
fan_id00 = pd.concat([fan_id00_abnormal, fan_id00_normal],ignore_index=True)
fan_id00['label'] = 0
fan_id02 = pd.concat([fan_id02_abnormal, fan_id02_normal],ignore_index=True)
fan_id02['label'] = 1
fan_id04 = pd.concat([fan_id04_abnormal, fan_id04_normal],ignore_index=True)
fan_id04['label'] = 2
fan_id06 = pd.concat([fan_id06_abnormal, fan_id06_normal],ignore_index=True)
fan_id06['label'] = 3

In [7]:
fan = pd.concat([fan_id00, fan_id02, fan_id04, fan_id06],ignore_index=True)

In [8]:
fan.describe()

,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,mfcc_3_mean,mfcc_3_std,mfcc_4_mean,mfcc_4_std,mfcc_5_mean,mfcc_5_std,...,spectral_bandwidth_mean,spectral_bandwidth_std,spectral_flatness_mean,spectral_flatness_std,spectral_rolloff_mean,rms_energy_mean,rms_energy_std,zcr_mean,zcr_std,label
count,5550.000000,5550.000000,5550.000000,5550.000000,5550.000000,5550.000000,5550.000000,5550.000000,5550.000000,5550.000000,...,5550.000000,5550.000000,5550.000000,5550.000000,5550.000000,5550.000000,5550.000000,5550.000000,5550.000000,5550.000000
mean,-366.354886,6.108918,174.096096,4.666179,-45.492614,4.189338,58.537018,3.889265,-32.580908,3.747553,...,1589.856554,56.171149,0.000291,0.000532,2849.083595,0.010078,0.001068,0.065302,0.009258,1.489189
std,19.458904,6.021730,13.524623,2.451523,13.025995,2.008252,8.086345,1.233069,9.421190,1.159895,...,188.871089,33.040926,0.000203,0.000412,656.852191,0.000955,0.000481,0.024601,0.006370,1.121141
min,-457.557920,2.841857,113.918370,2.644368,-110.434300,2.539403,31.599163,2.705487,-63.814700,2.691078,...,1044.785976,23.187554,0.000026,0.000024,1073.987241,0.005376,0.000377,0.017663,0.004352,0.000000
25%,-377.891512,3.585266,165.361435,3.604113,-51.423063,3.325662,52.818470,3.324676,-39.945470,3.249190,...,1433.020467,39.026854,0.000170,0.000262,2368.352578,0.009576,0.000790,0.050147,0.006479,0.000000
50%,-366.450345,3.981211,174.875925,3.979287,-45.505217,3.633870,58.772920,3.589252,-31.862337,3.445972,...,1600.486082,46.838244,0.000244,0.000425,2809.308612,0.010136,0.000925,0.061672,0.007349,1.000000
75%,-352.521417,5.458717,184.928843,4.653983,-37.808765,4.083744,63.527264,3.943126,-25.965122,3.752330,...,1726.584953,59.334155,0.000334,0.000670,3250.395794,0.010481,0.001151,0.073236,0.009306,2.000000
max,-315.884280,81.395744,207.038990,38.707530,-2.729767,29.531670,94.340890,18.758238,-1.728534,19.165861,...,2346.958216,539.525553,0.001903,0.005116,5616.668744,0.015144,0.006096,0.235644,0.087930,3.000000


In [11]:
X = fan.drop('label', axis=1)
y = fan['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101,stratify=y)

xgb_clf = xgb.XGBClassifier(
    objective='multi:softmax',
    eval_metric='mlogloss',
    random_state=101,
    device='cuda',
    n_jobs=1,

)

param_grid = {
    'n_estimators': [100, 200, 500, 700],
    'learning_rate': [0.01, 0.05, 0.1],
    'reg_alpha': [0.0, 0.1, 0.2, 0.5],
    'reg_lambda': [0.1, 0.2, 0.3, 0.5],
    'subsample': [0.5, 0.7, 0.8, 0.9],
    'gamma': [0.0, 0.1, 0.2, 0.5, 1],
}

search = RandomizedSearchCV(
    estimator=xgb_clf,
    param_distributions=param_grid,
    n_iter=100,
    scoring='roc_auc_ovr',
    cv=5,
    n_jobs=1,
    random_state=101,
    verbose=2,
    refit=True,
)

search.fit(X_train, y_train)


print("Best params:", search.best_params_)
print("Best CV score:", search.best_score_)

# Evaluate on test set
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)
print("\nTest accuracy:", accuracy_score(y_test, y_pred))
print("\n", classification_report(y_test, y_pred))

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV] END gamma=0.5, learning_rate=0.05, n_estimators=700, reg_alpha=0.2, reg_lambda=0.2, subsample=0.9; total time=   4.7s
[CV] END gamma=0.5, learning_rate=0.05, n_estimators=700, reg_alpha=0.2, reg_lambda=0.2, subsample=0.9; total time=   2.2s
[CV] END gamma=0.5, learning_rate=0.05, n_estimators=700, reg_alpha=0.2, reg_lambda=0.2, subsample=0.9; total time=   2.2s
[CV] END gamma=0.5, learning_rate=0.05, n_estimators=700, reg_alpha=0.2, reg_lambda=0.2, subsample=0.9; total time=   2.2s
[CV] END gamma=0.5, learning_rate=0.05, n_estimators=700, reg_alpha=0.2, reg_lambda=0.2, subsample=0.9; total time=   2.4s
[CV] END gamma=0.2, learning_rate=0.01, n_estimators=500, reg_alpha=0.2, reg_lambda=0.3, subsample=0.9; total time=   4.2s
[CV] END gamma=0.2, learning_rate=0.01, n_estimators=500, reg_alpha=0.2, reg_lambda=0.3, subsample=0.9; total time=   3.9s
[CV] END gamma=0.2, learning_rate=0.01, n_estimators=500, reg_alpha=0.2, reg

In [12]:
from joblib import dump, load
dump(best_model, 'fan_comp_model.joblib')

['fan_comp_model.joblib']